In [1]:
%pip install -q timm pydicom captum grad-cam scikit-image scipy seaborn

from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
PY = sys.executable


MODELS = ["convnext_blackbox", "cbm_nonleaky", "cbm_leaky", "resnet50", "densenet121", "efficientnet_b4", "vit_small"]
THEORY_MODELS = ["resnet50", "densenet121", "convnext_blackbox", "efficientnet_b4", "vit_small"]

CFG_NAME = {
    "convnext_blackbox": "configs/baselines/convnext_blackbox.yaml",
    "cbm_nonleaky": "configs/baselines/cbm_nonleaky.yaml",
    "cbm_leaky": "configs/baselines/cbm_leaky.yaml",
    "resnet50": "configs/baselines/resnet50.yaml",
    "densenet121": "configs/baselines/densenet121.yaml",
    "efficientnet_b4": "configs/baselines/efficientnet_b4.yaml",
    "vit_small": "configs/baselines/vit_small.yaml",
}
EXP = {
    "convnext_blackbox": "baseline_convnext_tiny_blackbox",
    "cbm_nonleaky": "cbm_nonleaky_7concepts",
    "cbm_leaky": "cbm_leaky_5concepts",
    "resnet50": "baseline_resnet50",
    "densenet121": "baseline_densenet121",
    "efficientnet_b4": "baseline_efficientnet_b4",
    "vit_small": "baseline_vit_small",
}

def run(cmd):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True)

def input_roots():
    roots = [WORK]
    if INPUT.exists():
        roots += [p for p in INPUT.iterdir() if p.is_dir()]
        datasets = INPUT / "datasets"
        if datasets.exists():
            for owner in datasets.iterdir():
                if owner.is_dir():
                    roots += [p for p in owner.iterdir() if p.is_dir()]
    return roots

def looks_like_code(p):
    return (
        (p / "scripts" / "train.py").exists()
        and (p / "configs").exists()
        and (p / "scripts" / "feature_map_smoothness.py").exists()
    )

def first_existing(candidates, label):
    for p in map(Path, candidates):
        if p.exists():
            return p
    preview = [str(p) for p in candidates[:10]]
    raise FileNotFoundError(f"Could not find {label}. Tried: {preview}")

def find_code_source():
    candidates = [INPUT / "spinexnet-code", INPUT / "spinexnet-code" / "het-spine"]
    for r in input_roots():
        candidates += [r, r / "het-spine", r / "spinexnet-code"]
    for p in candidates:
        if looks_like_code(p):
            return p
    raise FileNotFoundError("Could not find spinexnet-code")

def find_manifest():
    candidates = []
    for r in input_roots():
        candidates += [r / "manifest_v2.csv", r / "manifests" / "manifest_v2.csv"]
    return first_existing(candidates, "manifest_v2.csv")

def find_cache():
    candidates = []
    for r in input_roots():
        candidates += [r / "image_cache_224", r / "pre-processed-crop-224" / "image_cache_224"]
    candidates = [p for p in candidates if (p / "images_uint8.npy").exists()]
    return first_existing(candidates, "image_cache_224")

SRC = find_code_source()
CODE = WORK / "spinexnet-code"
if SRC.resolve() != CODE.resolve():
    shutil.copytree(SRC, CODE, dirs_exist_ok=True)
MANIFEST = find_manifest()
CACHE = find_cache()
CFG = {m: CODE / rel for m, rel in CFG_NAME.items()}

def find_checkpoint(model, fold):
    rels = [
        Path("outputs") / EXP[model] / f"fold_{fold}" / "best.pt",
        Path(EXP[model]) / f"fold_{fold}" / "best.pt",
        Path("checkpoints") / f"{model}_fold_{fold}_best.pt",
        Path("checkpoints") / f"{model}_fold{fold}_best.pt",
    ]
    if fold == 0:
        rels.append(Path("checkpoints") / f"{model}_best.pt")
    candidates = []
    for r in input_roots():
        candidates += [r / rel for rel in rels]
    candidates += [WORK / rel for rel in rels]
    return first_existing(candidates, f"{model} fold {fold} checkpoint")

def merge_tree_named(name):
    dest = WORK / f"combined_{name}"
    for r in input_roots():
        src = r / name
        if src.exists():
            try:
                if src.resolve() == dest.resolve():
                    continue
            except Exception:
                pass
            shutil.copytree(src, dest, dirs_exist_ok=True)
    return dest

EVAL_ROOT = merge_tree_named("eval_multifold")
XAI_ROOT = merge_tree_named("xai_multifold")
print("CODE:", CODE)
print("MANIFEST:", MANIFEST)
print("CACHE:", CACHE)
print("EVAL_ROOT:", EVAL_ROOT)
print("XAI_ROOT:", XAI_ROOT)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 107.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 28.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
CODE: /kaggle/working/spinexnet-code
MANIFEST: /kaggle/input/datasets/vasuaashadesai/manifests-of-spinexnet/manifests/manifest_v2.csv
CACHE: /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224
EVAL_ROOT: /kaggle/working/combined_eval_multifold
XAI_ROOT: /kaggle/working/combined_xai_multifold


In [2]:
import os, subprocess

os.environ["PYTHONPATH"] = f"{CODE}:{os.environ.get('PYTHONPATH', '')}"

def run(cmd):
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{CODE}:{env.get('PYTHONPATH', '')}"
    print("\n$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, cwd=CODE, env=env)

print("PYTHONPATH fixed:", os.environ["PYTHONPATH"].split(":")[0])

PYTHONPATH fixed: /kaggle/working/spinexnet-code


In [3]:

rows = []
for path in XAI_ROOT.glob("fold_*/*/xai_summary_v2.json"):
    with open(path) as f:
        d = json.load(f)["summary"]
    rows.append({
        "fold": path.parents[1].name,
        "model": path.parent.name,
        "mean_spearman": d.get("mean_spearman"),
        "mean_top20_iou": d.get("mean_top20_iou"),
        "consensus_insertion_auc_mean": d.get("consensus_insertion_auc_mean"),
        "consensus_expert_roi_mean": d.get("consensus_expert_roi_mean"),
    })

xai_summary = pd.DataFrame(rows)
if xai_summary.empty:
    print("No XAI summaries found.")
else:
    xai_summary["fold_idx"] = xai_summary["fold"].str.extract(r"(\d+)").astype(int)
    xai_summary = xai_summary.sort_values(["model", "fold_idx"]).drop(columns=["fold_idx"])
    xai_summary.to_csv(WORK / "xai_multifold_summary_by_fold.csv", index=False)
    xai_mean_std = xai_summary.groupby("model")[[
        "mean_spearman",
        "mean_top20_iou",
        "consensus_insertion_auc_mean",
        "consensus_expert_roi_mean",
    ]].agg(["mean", "std"]).round(4)
    xai_mean_std.to_csv(WORK / "xai_multifold_summary_mean_std.csv")
    display(xai_mean_std)


mean_spearman         mean_top20_iou          \
                           mean     std           mean     std   
model                                                            
cbm_leaky                0.3471  0.0647         0.3498  0.0383   
cbm_nonleaky             0.3798  0.0336         0.3451  0.0193   
convnext_blackbox        0.2826  0.0392         0.3257  0.0262   
densenet121              0.4340  0.0234         0.3849  0.0141   
efficientnet_b4          0.2232  0.0134         0.2307  0.0084   
resnet50                 0.4117  0.0226         0.3294  0.0152   
vit_small                0.1996  0.0281         0.2391  0.0206   

                  consensus_insertion_auc_mean          \
                                          mean     std   
model                                                    
cbm_leaky                               0.8004  0.0360   
cbm_nonleaky                            0.8263  0.0342   
convnext_blackbox                       0.8101  0.0196   
densenet121                             0.8079  0.0243   
efficientnet_b4                         0.7652  0.0731   
resnet50                                0.7875  0.0352   
vit_small                               0.7712  0.0304   

                  consensus_expert_roi_mean          
                                       mean     std  
model                                                
cbm_leaky                            0.2480  0.0167  
cbm_nonleaky                         0.2641  0.0173  
convnext_blackbox                    0.2448  0.0143  
densenet121                          0.2752  0.0019  
efficientnet_b4                      0.2597  0.0063  
resnet50                             0.2650  0.0082  
vit_small                            0.2095  0.0134

In [4]:

FIG = WORK / "figures"
XAI_FOLD0 = XAI_ROOT / "fold_0"

for model in MODELS:
    if not (XAI_FOLD0 / model / "xai_summary_v2.json").exists():
        print("Missing XAI fold_0 for", model)
        continue
    run([PY, CODE / "scripts/visualize_xai.py", "--results-dir", XAI_FOLD0 / model, "--output-dir", FIG / "fold_0" / model])

run([PY, CODE / "scripts/visualize_xai.py", "--cross-model-dir", XAI_FOLD0, "--output-dir", FIG / "fold_0" / "cross_model"])
run([PY, CODE / "scripts/generate_gallery.py", "--xai-dir", XAI_FOLD0, "--models", "densenet121", "convnext_blackbox", "vit_small", "--manifest", MANIFEST, "--cache-dir", CACHE, "--output-dir", FIG / "fold_0" / "gallery"])



$ /usr/bin/python3 /kaggle/working/spinexnet-code/scripts/visualize_xai.py --results-dir /kaggle/working/combined_xai_multifold/fold_0/convnext_blackbox --output-dir /kaggle/working/figures/fold_0/convnext_blackbox
Saved agreement heatmap to /kaggle/working/figures/fold_0/convnext_blackbox/agreement_heatmap.png
Saved faithfulness comparison to /kaggle/working/figures/fold_0/convnext_blackbox/faithfulness_comparison.png
Saved expert ROI alignment to /kaggle/working/figures/fold_0/convnext_blackbox/expert_roi_alignment.png
Saved clinical alignment to /kaggle/working/figures/fold_0/convnext_blackbox/clinical_alignment.png
Skipping consistency — /kaggle/working/combined_xai_multifold/fold_0/convnext_blackbox/consistency_metrics.csv not found

Per-model figures saved to /kaggle/working/figures/fold_0/convnext_blackbox

$ /usr/bin/python3 /kaggle/working/spinexnet-code/scripts/visualize_xai.py --results-dir /kaggle/working/combined_xai_multifold/fold_0/cbm_nonleaky --output-dir /kaggle/work

In [5]:

run([
    PY, CODE / "scripts/feature_map_smoothness.py",
    "--configs", *[CFG[m] for m in THEORY_MODELS],
    "--checkpoints", *[find_checkpoint(m, 0) for m in THEORY_MODELS],
    "--names", *THEORY_MODELS,
    "--manifest", MANIFEST,
    "--fold", 0,
    "--cache-dir", CACHE,
    "--output-dir", WORK / "feature_map_smoothness",
    "--max-samples", 300,
    "--agreement-csv", FIG / "fold_0" / "cross_model" / "cross_model_agreement.csv",
])



$ /usr/bin/python3 /kaggle/working/spinexnet-code/scripts/feature_map_smoothness.py --configs /kaggle/working/spinexnet-code/configs/baselines/resnet50.yaml /kaggle/working/spinexnet-code/configs/baselines/densenet121.yaml /kaggle/working/spinexnet-code/configs/baselines/convnext_blackbox.yaml /kaggle/working/spinexnet-code/configs/baselines/efficientnet_b4.yaml /kaggle/working/spinexnet-code/configs/baselines/vit_small.yaml --checkpoints /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/baseline_resnet50/fold_0/best.pt /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/baseline_densenet121/fold_0/best.pt /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/baseline_convnext_tiny_blackbox/fold_0/best.pt /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/baseline_efficientnet_b4/fold_0/best.pt /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-model

{'model': 'resnet50', 'feature_autocorrelation': 0.6476, 'feature_total_variation': 0.5562, 'feature_coherence_score': 0.4168}

Computing feature-map coherence for densenet121


{'model': 'densenet121', 'feature_autocorrelation': 0.6762, 'feature_total_variation': 0.5748, 'feature_coherence_score': 0.4308}

Computing feature-map coherence for convnext_blackbox


{'model': 'convnext_blackbox', 'feature_autocorrelation': 0.3142, 'feature_total_variation': 0.5219, 'feature_coherence_score': 0.2049}

Computing feature-map coherence for efficientnet_b4


{'model': 'efficientnet_b4', 'feature_autocorrelation': -0.0093, 'feature_total_variation': 0.3876, 'feature_coherence_score': -0.0079}

Computing feature-map coherence for vit_small


Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.


{'model': 'vit_small', 'feature_autocorrelation': 0.6873, 'feature_total_variation': 0.5209, 'feature_coherence_score': 0.4586}
Saved feature coherence scatter plot to /kaggle/working/feature_map_smoothness/feature_coherence_vs_agreement.png

Feature-map smoothness results saved to /kaggle/working/feature_map_smoothness


In [6]:

for model in ["convnext_blackbox", "vit_small"]:
    run([
        PY, CODE / "scripts/model_randomization.py",
        "--config", CFG[model],
        "--checkpoint", find_checkpoint(model, 0),
        "--manifest", MANIFEST,
        "--fold", 0,
        "--cache-dir", CACHE,
        "--output-dir", WORK / "randomization" / model,
        "--methods", "gradcam", "integrated_gradients", "gradient_shap", "occlusion",
        "--max-samples", 50,
        "--n-levels", 5,
    ])

for model in ["cbm_nonleaky", "cbm_leaky"]:
    run([
        PY, CODE / "scripts/concept_intervention.py",
        "--config", CFG[model],
        "--checkpoint", find_checkpoint(model, 0),
        "--manifest", MANIFEST,
        "--fold", 0,
        "--cache-dir", CACHE,
        "--output-dir", WORK / "intervention" / model,
        "--max-samples", 1000,
    ])



$ /usr/bin/python3 /kaggle/working/spinexnet-code/scripts/model_randomization.py --config /kaggle/working/spinexnet-code/configs/baselines/convnext_blackbox.yaml --checkpoint /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/baseline_convnext_tiny_blackbox/fold_0/best.pt --manifest /kaggle/input/datasets/vasuaashadesai/manifests-of-spinexnet/manifests/manifest_v2.csv --fold 0 --cache-dir /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224 --output-dir /kaggle/working/randomization/convnext_blackbox --methods gradcam integrated_gradients gradient_shap occlusion --max-samples 50 --n-levels 5
[RSNACropDataset] Using image cache /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224/images_uint8.npy with shape (48657, 224, 224).


randomization: 100%|██████████| 50/50 [31:49<00:00, 38.18s/it]


Saved randomization plot to /kaggle/working/randomization/convnext_blackbox/randomization_plot.png
Randomization results saved to /kaggle/working/randomization/convnext_blackbox

$ /usr/bin/python3 /kaggle/working/spinexnet-code/scripts/model_randomization.py --config /kaggle/working/spinexnet-code/configs/baselines/vit_small.yaml --checkpoint /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/baseline_vit_small/fold_0/best.pt --manifest /kaggle/input/datasets/vasuaashadesai/manifests-of-spinexnet/manifests/manifest_v2.csv --fold 0 --cache-dir /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224 --output-dir /kaggle/working/randomization/vit_small --methods gradcam integrated_gradients gradient_shap occlusion --max-samples 50 --n-levels 5
[RSNACropDataset] Using image cache /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224/images_uint8.npy with shape (48657, 224, 224).


Unexpected keys (norm.bias, norm.weight) found while loading pretrained weights. This may be expected if model is being adapted.
randomization: 100%|██████████| 50/50 [28:55<00:00, 34.71s/it]


Saved randomization plot to /kaggle/working/randomization/vit_small/randomization_plot.png
Randomization results saved to /kaggle/working/randomization/vit_small

$ /usr/bin/python3 /kaggle/working/spinexnet-code/scripts/concept_intervention.py --config /kaggle/working/spinexnet-code/configs/baselines/cbm_nonleaky.yaml --checkpoint /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/cbm_nonleaky_7concepts/fold_0/best.pt --manifest /kaggle/input/datasets/vasuaashadesai/manifests-of-spinexnet/manifests/manifest_v2.csv --fold 0 --cache-dir /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224 --output-dir /kaggle/working/intervention/cbm_nonleaky --max-samples 1000
[RSNACropDataset] Using image cache /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224/images_uint8.npy with shape (48657, 224, 224).


intervention: 100%|██████████| 1000/1000 [00:12<00:00, 82.43it/s] 


{'total_samples': 1000, 'correct_before_intervention': 848, 'accuracy_before': 0.848, 'wrong_samples': 152, 'fixable_by_any_concept': 90, 'fix_rate': 0.5921, 'per_concept_fix_rate': {'concept_nonleaky_adjacent_pathology_density': 0.5132, 'concept_nonleaky_is_foraminal': 0.0, 'concept_nonleaky_is_stenosis': 0.0, 'concept_nonleaky_is_subarticular': 0.0, 'concept_nonleaky_left_laterality': 0.0066, 'concept_nonleaky_level_position': 0.125, 'concept_nonleaky_right_laterality': 0.0}}

$ /usr/bin/python3 /kaggle/working/spinexnet-code/scripts/concept_intervention.py --config /kaggle/working/spinexnet-code/configs/baselines/cbm_leaky.yaml --checkpoint /kaggle/input/datasets/shilpavdesai/train-eval-xai-efficientnet-b4-models/outputs/cbm_leaky_5concepts/fold_0/best.pt --manifest /kaggle/input/datasets/vasuaashadesai/manifests-of-spinexnet/manifests/manifest_v2.csv --fold 0 --cache-dir /kaggle/input/datasets/vasuaashadesai/pre-processed-crop-224/image_cache_224 --output-dir /kaggle/working/interv

intervention: 100%|██████████| 1000/1000 [00:12<00:00, 82.51it/s]


{'total_samples': 1000, 'correct_before_intervention': 838, 'accuracy_before': 0.838, 'wrong_samples': 162, 'fixable_by_any_concept': 137, 'fix_rate': 0.8457, 'per_concept_fix_rate': {'concept_pseudo_adjacent_pathology_density': 0.3704, 'concept_pseudo_left_laterality': 0.0, 'concept_pseudo_pathology_present': 0.7407, 'concept_pseudo_right_laterality': 0.0, 'concept_pseudo_severe_grade': 0.1605}}
